In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [7]:
prompt = ChatPromptTemplate.from_template(
    'Reply in one sentence: {question}'
)

parser = StrOutputParser()
chain = prompt | llm 

In [9]:
chain.invoke('hi').content

'Hello! How can I assist you today?'

In [10]:
classify_prompt = ChatPromptTemplate.from_template(
    'Classify this email in ONE word (Billing/Technical/Spam/Other):\n'
    'Subject: {subject}\nBody: {body}'
)

clasify_chain = classify_prompt | llm | parser

In [11]:
emails = [
    {'subject': 'Payment failed',   'body': 'Charged twice this month.'},
    {'subject': 'App crash',        'body': 'Crashes on Settings page.'},
    {'subject': 'Add Slack?',       'body': 'Want Slack notifications.'},
    {'subject': 'WIN FREE IPHONE',  'body': 'Click here NOW!!!'},
    {'subject': 'Invoice question', 'body': 'Can I get annual invoice?'},
]

batch_results = clasify_chain.batch(emails,
                                    config={'max_concurrency':3})

In [12]:
batch_results

['Billing', 'Technical', 'Other', 'Spam', 'Billing']

In [13]:
# lets suppose you have your chain -->

outputs_from_llm = [
    'CATEGORY: Billing\nURGENCY: High',        # normal
    'CATEGORY:  Billing\nURGENCY: High',       # extra space!
    'Category: Billing\nUrgency: High',         # different case!
    'CATEGORY: Billing (payment issue)\nURGENCY: High',  # extra text!
    'I think this is CATEGORY: Billing\nURGENCY: High',  # explanation first!
]

print('Parsing with split() — watch it break:')
print('─' * 60)

for output in outputs_from_llm:
    category = 'PARSE FAILED'
    for line in output.split('\n'):
        if 'CATEGORY:' in line:  # exact match — brittle!
            category = line.replace('CATEGORY:', '').strip()
            break
    print(f'Input: {output[:40]:40} → Category: "{category}"')

Parsing with split() — watch it break:
────────────────────────────────────────────────────────────
Input: CATEGORY: Billing
URGENCY: High          → Category: "Billing"
Input: CATEGORY:  Billing
URGENCY: High         → Category: "Billing"
Input: Category: Billing
Urgency: High          → Category: "PARSE FAILED"
Input: CATEGORY: Billing (payment issue)
URGENC → Category: "Billing (payment issue)"
Input: I think this is CATEGORY: Billing
URGENC → Category: "I think this is  Billing"


In [14]:
# pydantic base model ( baemodel)

# let me define my output sturucture

from pydantic import BaseModel, Field
from typing import Literal, List

class EmailClassification(BaseModel):
    category : Literal[
        'Billing',
        'Technical', 
        'Feature Request',
        'Churn Risk',
        'Spam',
        'Other'
    ] = Field(description="The category of the support email")

    urgency: Literal['High', 'Medium', 'Low'] = Field(
        description='Urgency based on business impact'
    )
    confidence: int = Field(
        description='Confidence score from 1 to 10',
        ge=1,   # ge = greater than or equal (Pydantic v2!)
        le=10   # le = less than or equal
    )
    reasoning: str = Field(
        description='One sentence explanation of the classification'
    )

    cot_steps: List[str] = Field(
        description='3-5 Chain of Thought reasoning steps'
    )




In [21]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=EmailClassification)

In [24]:
print(parser.get_format_instructions()) # kind of text out put --> willl be read your lllm so he can understand how my output should look like

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"category": {"description": "The category of the support email", "enum": ["Billing", "Technical", "Feature Request", "Churn Risk", "Spam", "Other"], "title": "Category", "type": "string"}, "urgency": {"description": "Urgency based on business impact", "enum": ["High", "Medium", "Low"], "title": "Urgency", "type": "string"}, "confidence": {"description": "Confidence score from 1 to 10", "maximum": 10, "minimum": 1, "title": "Confidence", "type": "integer"}, "reasoning": {"description": "One sentence explanation of the classifica

In [25]:

cot_prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are an expert support email classifier.

Classification Rules:
- Login broken after payment → Billing (NOT Technical)
- App crashes → Technical
- Pricing complaints + evaluating alternatives → Churn Risk
- Feature requests → Feature Request

Think step by step before classifying.

{format_instructions}'''),

    ('human', 'Subject: {subject}\nBody: {body}')
]).partial(
    format_instructions = parser.get_format_instructions()
)

In [27]:
# define our chain

cot_chain = cot_prompt | llm | parser

In [28]:
# Step 5: USE the chain!
email = {
    'subject': "Can't login — paid for annual plan last week",
    'body': 'Our entire team cannot login. We paid for the annual plan '
            'last week. We have a board demo in 3 hours!'
}

print('Testing chain...')
result = cot_chain.invoke(email)
result

Testing chain...


EmailClassification(category='Billing', urgency='High', confidence=9, reasoning='The issue relates to a login problem following a payment, which categorizes it under Billing.', cot_steps=['Identify the issue as a login problem.', 'Note that the user has made a payment for the annual plan.', 'Recognize the urgency due to an upcoming demo in 3 hours.', 'Apply classification rules, determining that this is not a technical issue but a billing-related issue.'])

In [30]:
print(f'  result.category   = "{result.category}"')
print(f'  result.urgency    = "{result.urgency}"')
print(f'  result.confidence = {result.confidence}  ← int, validated!')
print(f'  result.reasoning  = "{result.reasoning[:60]}..."')

  result.category   = "Billing"
  result.urgency    = "High"
  result.confidence = 9  ← int, validated!
  result.reasoning  = "The issue relates to a login problem following a payment, wh..."
